# 📘 File 07.Full Project Dashboard

In [1]:
import dash
from dash import dcc, html, dash_table
import plotly.express as px
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load Data
# -----------------------------
calls_raw = pd.read_excel("Calls (CRM).xlsx")
contacts_raw = pd.read_excel("Contacts (CRM).xlsx")
deals_raw = pd.read_excel("Deals (CRM).xlsx")
spend_raw = pd.read_excel("Spend (CRM).xlsx")

calls = pd.read_excel("Calls (Result).xlsx")
contacts = pd.read_excel("Contacts (Result).xlsx")
deals = pd.read_excel("Deals_clean.xlsx")
spend = pd.read_excel("Spend (Result).xlsx")

# -----------------------------
# 2. Cleaning: File Sizes
# -----------------------------
def df_info(df, name):
    return pd.DataFrame({"File": [name], "Rows": [df.shape[0]], "Columns": [df.shape[1]]})

raw_info = pd.concat([
    df_info(calls_raw, "Calls (Done)"),
    df_info(contacts_raw, "Contacts (Done)"),
    df_info(deals_raw, "Deals (Done)"),
    df_info(spend_raw, "Spend (Done)")
], ignore_index=True)

clean_info = pd.concat([
    df_info(calls, "Calls (Result)"),
    df_info(contacts, "Contacts (Result)"),
    df_info(deals, "Deals (Clean)"),
    df_info(spend, "Spend (Result)")
], ignore_index=True)

size_compare = pd.DataFrame({
    "File": ["Calls", "Contacts", "Deals", "Spend"],
    "Rows_before": raw_info["Rows"],
    "Columns_before": raw_info["Columns"],
    "Rows_after": clean_info["Rows"],
    "Columns_after": clean_info["Columns"],
})

raw_info["Empty_before"] = [
    calls_raw.isna().sum().sum(),
    contacts_raw.isna().sum().sum(),
    deals_raw.isna().sum().sum(),
    spend_raw.isna().sum().sum()
]

clean_info["Empty_after"] = [
    calls.isna().sum().sum(),
    contacts.isna().sum().sum(),
    deals.isna().sum().sum(),
    spend.isna().sum().sum()
]

size_compare = pd.DataFrame({
    "File": ["Calls", "Contacts", "Deals", "Spend"],
    "Rows_before": raw_info["Rows"],
    "Columns_before": raw_info["Columns"],
    "Empty_before": raw_info["Empty_before"],
    "Rows_after": clean_info["Rows"],
    "Columns_after": clean_info["Columns"],
    "Empty_after": clean_info["Empty_after"]
})

# -----------------------------
# 3. Dates & Basic Fields
# -----------------------------
deals["Created Date"] = pd.to_datetime(deals["Created Date"], errors="coerce")
deals["month"] = deals["Created Date"].dt.to_period("M")

# -----------------------------
# 4. Unit Economics
# -----------------------------
MAIN_PRODUCTS = ["Digital Marketing", "UX/UI Design", "Web Developer"]
valid_edu = ["Morning", "Evening"]

deals_pe = deals[
    deals["Product"].isin(MAIN_PRODUCTS) &
    deals["Education Type"].isin(valid_edu)
].copy()

payment_done = deals_pe[deals_pe["Stage"] == "Payment Done"].copy()

UA = deals_pe.groupby(["Product", "Education Type"])["Contact Name"].nunique().rename("UA")
B = payment_done.groupby(["Product", "Education Type"])["Id"].count().rename("B")
T = payment_done.groupby(["Product", "Education Type"])["Months of study"].sum().rename("T")
Revenue = payment_done.groupby(["Product", "Education Type"])["Revenue"].sum().rename("Revenue")

AC_value = spend["Spend"].sum()

df_pe = (
    UA.to_frame()
    .join([B, T, Revenue], how="left")
    .reset_index()
)

df_pe["AC"] = AC_value
df_pe["COGS"] = 0

df_pe["C1"] = df_pe["B"] / df_pe["UA"]
df_pe["CPA"] = df_pe["AC"] / df_pe["UA"]
df_pe["CAC"] = df_pe["AC"] / df_pe["B"]
df_pe["AOV"] = df_pe["Revenue"] / df_pe["T"]
df_pe["APC"] = df_pe["T"] / df_pe["B"]
df_pe["GP"] = df_pe["AOV"] * df_pe["T"]
df_pe["CLTV"] = df_pe["GP"] / df_pe["B"]
df_pe["LTV"] = df_pe["CLTV"] * df_pe["C1"]
df_pe["CM"] = (df_pe["CLTV"] - df_pe["CAC"]) * df_pe["B"]

df_sum = df_pe.groupby("Product").agg({
    "UA": "sum",
    "B": "sum",
    "T": "sum",
    "Revenue": "sum",
    "AC": "mean"
}).reset_index()

df_sum["COGS"] = 0

UA_total = 18548
B_sum = df_sum["B"].sum()

df_sum["UA"] = df_sum["B"] / B_sum * UA_total
df_sum["C1"] = df_sum["B"] / df_sum["UA"]
df_sum["CPA"] = df_sum["AC"] / df_sum["UA"]
df_sum["CAC"] = df_sum["AC"] / df_sum["B"]
df_sum["AOV"] = df_sum["Revenue"] / df_sum["T"]
df_sum["APC"] = df_sum["T"] / df_sum["B"]
df_sum["GP"] = df_sum["AOV"] * df_sum["T"]
df_sum["CLTV"] = df_sum["GP"] / df_sum["B"]
df_sum["LTV"] = df_sum["CLTV"] * df_sum["C1"]
df_sum["CM"] = (df_sum["CLTV"] - df_sum["CAC"]) * df_sum["B"]

# -----------------------------
# 5. TOTAL Row
# -----------------------------
total_row = pd.DataFrame([{
    "Product": "TOTAL",
    "UA": df_sum["UA"].sum(),
    "B": df_sum["B"].sum(),
    "T": df_sum["T"].sum(),
    "Revenue": df_sum["Revenue"].sum(),
    "AC": df_sum["AC"].mean(),
    "COGS": 0
}])

total_row["C1"] = total_row["B"] / total_row["UA"]
total_row["CPA"] = total_row["AC"] / total_row["UA"]
total_row["CAC"] = total_row["AC"] / total_row["B"]
total_row["AOV"] = total_row["Revenue"] / total_row["T"]
total_row["APC"] = total_row["T"] / total_row["B"]
total_row["GP"] = total_row["AOV"] * total_row["T"]
total_row["CLTV"] = total_row["GP"] / total_row["B"]
total_row["LTV"] = total_row["CLTV"] * total_row["C1"]
total_row["CM"] = (total_row["CLTV"] - total_row["CAC"]) * total_row["B"]

df_sum = pd.concat([df_sum, total_row], ignore_index=True)

# -----------------------------
# 6. KPI Tab
# -----------------------------
kpi_transactions = df_sum[df_sum["Product"] == "TOTAL"]["T"].iloc[0]
best_row = df_pe.sort_values("B", ascending=False).iloc[0]
best_product = best_row["Product"]
best_edu_type = best_row["Education Type"]
avg_check = df_sum[df_sum["Product"] == "TOTAL"]["AOV"].iloc[0]

kpi_calls = calls["Id"].count()
kpi_deals = deals[
    (deals["Stage"] == "Payment Done") &
    (deals["Product"].isin(MAIN_PRODUCTS)) &
    (deals["Education Type"].isin(valid_edu))
]["Id"].count()
kpi_managers = deals["Deal Owner Name"].nunique()
kpi_students_year = payment_done["Contact Name"].nunique()

kpi_data = pd.DataFrame({
    "Metric": [
        "Transactions (T)",
        "Best Product",
        "Education Type",
        "Average Check (AOV)",
        "Number of Calls",
        "Number of Deals",
        "Number of Managers",
        "Students per Year"
    ],
    "Value": [
        kpi_transactions,
        best_product,
        best_edu_type,
        round(avg_check, 2),
        kpi_calls,
        kpi_deals,
        kpi_managers,
        kpi_students_year
    ]
})

# -----------------------------
# 7. Deals by Month
# -----------------------------
trend_month = deals.groupby("month")["Id"].nunique().reset_index()
trend_month["month"] = trend_month["month"].astype(str)
trend_month = trend_month.sort_values("Id", ascending=False)

fig_deals_month = px.bar(
    trend_month,
    x="month",
    y="Id",
    title="Deals by Month",
    text="Id",
    color="Id",
    color_continuous_scale="Viridis"
)
fig_deals_month.update_traces(textposition="outside")

# -----------------------------
# 8. Sales Department
# -----------------------------
owners_perf = deals.groupby("Deal Owner Name").agg({
    "Id": "count",
    "Revenue": "sum",
    "Is_won": "sum"
}).reset_index()

owners_perf["CR"] = owners_perf["Is_won"] / owners_perf["Id"]
owners_perf = owners_perf.sort_values("Revenue", ascending=False)

fig_sales = px.bar(
    owners_perf,
    x="Revenue",
    y="Deal Owner Name",
    title="Managers by Revenue",
    color="Revenue",
    color_continuous_scale="Bluered"
)

fig_sales.update_layout(
    yaxis=dict(categoryorder="category ascending")
)

# -----------------------------
# 9. Geography & German Level
# -----------------------------
deals_df = pd.read_excel("Deals_clean_geo.xlsx")

city_perf = (
    deals_df.groupby("City_clean")
    .agg(
        Deals_count=("Id", "count"),
        Deals_won=("Is_won", "sum"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
)

city_perf["CR"] = city_perf["Deals_won"] / city_perf["Deals_count"]
city_perf["AOV"] = city_perf["Revenue"] / city_perf["Deals_won"].replace(0, np.nan)
city_perf = city_perf.sort_values("Deals_won", ascending=False).round(2)

fig_city = px.bar(
    city_perf.head(15),
    x="Deals_won",
    y="City_clean",
    title="Top Cities by Successful Deals",
    color="Deals_won",
    color_continuous_scale="Plasma"
)

fig_city.update_layout(
    yaxis=dict(categoryorder="category ascending")
)

lang_perf = (
    deals_df.groupby("Level of Deutsch")
    .agg(
        Deals_count=("Id", "count"),
        Deals_won=("Is_won", "sum")
    )
    .reset_index()
)

lang_perf = lang_perf[
    ~lang_perf["Level of Deutsch"].isin(["Unknown", "#REF!", "", None, "-"])
]

lang_perf["CR"] = lang_perf["Deals_won"] / lang_perf["Deals_count"]
lang_perf = lang_perf.sort_values("CR", ascending=False)

fig_lang = px.bar(
    lang_perf,
    x="CR",
    y="Level of Deutsch",
    title="CR by German Level",
    color="CR",
    color_continuous_scale="Viridis"
)

fig_lang.update_layout(
    yaxis=dict(categoryorder="category ascending")
)

# -----------------------------
# 10. Sales Funnel
# -----------------------------
funnel_sales = pd.DataFrame({
    "Stage": ["Deals", "Transactions (T)", "Calls", "Clicks"],
    "Count": [
        deals[deals["Stage"] == "Payment Done"]["Id"].count(),
        df_sum[df_sum["Product"] == "TOTAL"]["T"].iloc[0],
        calls["Id"].count(),
        spend["Clicks"].sum()
    ]
})

fig_funnel = px.funnel(
    funnel_sales,
    y="Stage",
    x="Count",
    title="Deals → Transactions → Calls → Clicks",
    color="Stage",
    color_discrete_sequence=px.colors.qualitative.Set2
)

# -----------------------------
# 11. HADI Simulator
# -----------------------------
base_revenue = df_sum[df_sum["Product"] == "TOTAL"]["Revenue"].iloc[0]

hadi_df = pd.DataFrame({
    "Scenario": ["H2", "H4", "H3", "H5", "H1"],
    "Revenue_new": [
        base_revenue * 1.20,
        base_revenue * 1.20,
        base_revenue * 1.15,
        base_revenue * 1.15,
        base_revenue * 1.10
    ]
})

hadi_df["Revenue_new"] = hadi_df["Revenue_new"].round(2)
hadi_df = hadi_df.sort_values("Revenue_new", ascending=False)

fig_hadi = px.bar(
    hadi_df,
    x="Scenario",
    y="Revenue_new",
    text="Revenue_new",
    title="HADI Simulator",
    color="Revenue_new",
    color_continuous_scale="Viridis"
)

fig_hadi.update_traces(textposition="inside")
fig_hadi.update_layout(
    xaxis=dict(categoryorder="array", categoryarray=hadi_df["Scenario"])
)

# -----------------------------
# 12. Dash Layout
# -----------------------------
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Full Project Dashboard", style={"textAlign": "center"}),

    dcc.Tabs([

        dcc.Tab(label="KPI", children=[
            html.Div([
                html.H2("Key Performance Indicators", style={
                    "textAlign": "center",
                    "marginBottom": "20px",
                    "fontSize": "32px"
                }),

                dash_table.DataTable(
                    data=kpi_data.to_dict("records"),
                    columns=[{"name": c, "id": c} for c in kpi_data.columns],
                    page_size=10,
                    style_table={
                        "margin": "0 auto",
                        "width": "60%",
                        "border": "1px solid #ccc",
                        "borderRadius": "10px"
                    },
                    style_cell={
                        "textAlign": "center",
                        "fontSize": "20px",
                        "padding": "12px"
                    },
                    style_header={
                        "backgroundColor": "#f0f0f0",
                        "fontWeight": "bold",
                        "fontSize": "22px"
                    }
                )
            ])
        ]),

        dcc.Tab(label="Data Cleaning", children=[
            dash_table.DataTable(
                data=size_compare.to_dict("records"),
                columns=[{"name": c, "id": c} for c in size_compare.columns],
                page_size=10,
                style_table={"overflowX": "auto"}
            )
        ]),

        dcc.Tab(label="Deals by Month", children=[
            dcc.Graph(figure=fig_deals_month)
        ]),

        dcc.Tab(label="Sales Department", children=[
            dcc.Graph(figure=fig_sales)
        ]),

        dcc.Tab(label="Geography & German Level", children=[
            dcc.Graph(figure=fig_city),
            dcc.Graph(figure=fig_lang)
        ]),

        dcc.Tab(label="Unit Economics", children=[
            dash_table.DataTable(
                data=df_sum.round(2).to_dict("records"),
                columns=[{"name": c, "id": c} for c in df_sum.columns],
                page_size=10,
                style_table={"overflowX": "auto"}
            )
        ]),

        dcc.Tab(label="Sales Funnel", children=[
            dcc.Graph(figure=fig_funnel)
        ]),

        dcc.Tab(label="HADI Simulator", children=[
            dcc.Graph(figure=fig_hadi)
        ])

    ])
])

# -----------------------------
# 13. Run
# -----------------------------
if __name__ == "__main__":
    app.run(debug=True, port=8051)